# Sudoku board line experiment

Notebook sluzy tylko do szybkiego sterowania eksperymentem i wizualizacji. Logika adaptive threshold, scalania linii i debug overlay siedzi w modulach `.py` obok.

In [ ]:
%matplotlib inline

from pathlib import Path
import importlib
import sys

import cv2
import matplotlib.pyplot as plt
import numpy as np

DRAFT_DIR = Path("/home/wojtek/projects/sudoku/src/MachineLearning/draft")
if str(DRAFT_DIR) not in sys.path:
    sys.path.insert(0, str(DRAFT_DIR))

import sudoku_board_debug_binary_variants as debug_binary_variants
import sudoku_board_debug_line_experiment as debug_line_experiment
import sudoku_board_debug_notebook as debug_notebook
import sudoku_board_debug_preprocess as debug_preprocess
import sudoku_board_debug_visualization as debug_visualization

debug_binary_variants = importlib.reload(debug_binary_variants)
debug_line_experiment = importlib.reload(debug_line_experiment)
debug_visualization = importlib.reload(debug_visualization)
debug_notebook = importlib.reload(debug_notebook)
debug_preprocess = importlib.reload(debug_preprocess)

BinaryCleanupSettings = debug_binary_variants.BinaryCleanupSettings
build_binary_variants = debug_binary_variants.build_binary_variants

LineMergeSettings = debug_line_experiment.LineMergeSettings
resolve_line_merge_settings = debug_line_experiment.resolve_line_merge_settings
run_line_detection_experiment = debug_line_experiment.run_line_detection_experiment
summarize_line_experiment = debug_line_experiment.summarize_line_experiment

build_notebook_debug_view = debug_notebook.build_notebook_debug_view
prepare_board_debug_image = debug_notebook.prepare_board_debug_image
print_cleanup_settings = debug_notebook.print_cleanup_settings
print_line_experiment_report = debug_notebook.print_line_experiment_report
print_resolved_line_settings = debug_notebook.print_resolved_line_settings

BoardDebugSettings = debug_preprocess.BoardDebugSettings
show_image = debug_visualization.show_image

plt.rcParams["figure.figsize"] = (18, 10)
plt.rcParams["image.cmap"] = "gray"


In [ ]:
IMAGE_PATH = Path("/home/wojtek/projects/sudoku/examples/uploads/image1041.jpg")

board_settings = BoardDebugSettings(
    grayscale_color_conversion_code=cv2.COLOR_BGR2GRAY,
    gaussian_kernel_size=(5, 5),
    gaussian_sigma_x=0.0,
    adaptive_threshold_block_size=11,
    adaptive_threshold_c=2,
)
line_settings = LineMergeSettings()
cleanup_settings = BinaryCleanupSettings()

print(f"IMAGE_PATH = {IMAGE_PATH}")
print("Notebook skupia sie tylko na znajdowaniu linii po adaptive threshold.")


In [ ]:
prepared_image = prepare_board_debug_image(IMAGE_PATH, board_settings)
source_image = prepared_image.source_image
preprocessed_image = prepared_image.preprocessed_image
binary_image = prepared_image.binary_image
binary_display_image = prepared_image.binary_display_image
minimum_dimension = prepared_image.minimum_dimension

resolved_line_settings = resolve_line_merge_settings(binary_image.shape, line_settings)
print_resolved_line_settings(resolved_line_settings, prepared_image)

figure, axes = plt.subplots(1, 3, figsize=(18, 6))
show_image(axes[0], source_image, "Source image", is_bgr=True)
show_image(axes[1], preprocessed_image, "Grayscale + blur")
show_image(axes[2], binary_display_image, "Adaptive threshold")
figure.tight_layout()


## Eksperyment bazowy

Najpierw sprawdzamy czysty wariant `adaptive only`, zeby zobaczyc surowe segmenty Hough, rodziny katow i wynik po obu etapach scalania kandydatow.

In [ ]:
base_result = run_line_detection_experiment(
    "adaptive only",
    binary_image,
    line_settings,
)
base_debug_view = build_notebook_debug_view(source_image, base_result)

print_line_experiment_report(
    base_result,
    title="Line finding experiment",
)

figure, axes = plt.subplots(2, 4, figsize=(28, 12))
show_image(axes[0, 0], source_image, "Source image", is_bgr=True)
show_image(axes[0, 1], preprocessed_image, "Grayscale + blur")
show_image(axes[0, 2], base_debug_view.binary_display_image, "Adaptive threshold (display)")
show_image(axes[0, 3], base_debug_view.raw_segments_overlay, "Raw Hough segments", is_bgr=True)
show_image(axes[1, 0], base_debug_view.family_overlay, "Two dominant angle families", is_bgr=True)
show_image(axes[1, 1], base_debug_view.filtered_overlay, "After segment merge", is_bgr=True)
show_image(axes[1, 2], base_debug_view.final_overlay, "After candidate post-merge", is_bgr=True)
show_image(axes[1, 3], base_debug_view.final_overlay_on_source, "Final candidates on source", is_bgr=True)
figure.tight_layout()


## Eksperyment: czyszczenie binarki przed szukaniem linii

Porownujemy kilka wariantow przygotowania obrazu po `adaptive threshold`, zanim uruchomimy Hough i dalsze scalanie kandydatow linii.

In [ ]:
resolved_cleanup_settings, line_variants = build_binary_variants(
    binary_image,
    minimum_dimension,
    cleanup_settings,
)
variant_results = [
    run_line_detection_experiment(name, variant_binary, line_settings)
    for name, variant_binary in line_variants
]
variant_summaries = [summarize_line_experiment(result) for result in variant_results]

print_cleanup_settings(resolved_cleanup_settings)
for result, summary in zip(variant_results, variant_summaries):
    print()
    print_line_experiment_report(
        result,
        title=f"Variant: {result.name}",
        include_candidate_details=False,
    )
    print(
        "Target >= 10 x 10:",
        "YES" if summary.has_target_grid else "NO",
    )

figure, axes = plt.subplots(len(variant_results), 1, figsize=(8, 6 * len(variant_results)))
axes = np.atleast_1d(axes)

for row_index, result in enumerate(variant_results):
    debug_view = build_notebook_debug_view(source_image, result)
    show_image(
        axes[row_index],
        debug_view.family_overlay,
        f"{result.name}\nangle families",
        is_bgr=True,
    )

figure.tight_layout()
